# ICT playbook analyst
Drop chart screenshots in a Google Drive folder, press **Run**, and get a trade card: **direction, entry, stop loss, take profit** - or "no trade" and why.

**How it works:** candle data is downloaded automatically (Yahoo Finance); Python computes the levels, sessions and patterns; Claude reads that, your screenshots, the economic calendar CSVs from your folder and your playbook (a Google Doc); Python then checks and sizes the idea against your risk rules.

**Drive folders (created for you):** `MyDrive/ict/screenshots` (your chart images), `MyDrive/ict/economic` (economic calendar CSV exports), `MyDrive/ict/data` (optional CSV exports - used instead of Yahoo if present), `MyDrive/ict/ideas` (saved results), `MyDrive/ict/journal.json` (optional closed-trade log).

*Educational analysis only - not financial advice. It never places orders.*

## 1. Setup (once per session)

In [ ]:
import os, sys
if not os.path.isdir("/content/Coding-Projects-"):
    !git clone -q https://github.com/EliseoRiddick/Coding-Projects-.git
else:
    !git -C /content/Coding-Projects- pull -q
!pip install -q anthropic pillow yfinance
sys.path.insert(0, "/content/Coding-Projects-/ict-playbook/agent")

from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")   # left sidebar > key icon > add secret, enable notebook access
try:   # optional: only needed when your API key is not tied to a workspace (add a Colab secret named ANTHROPIC_WORKSPACE_ID)
    os.environ["ANTHROPIC_WORKSPACE_ID"] = userdata.get("ANTHROPIC_WORKSPACE_ID")
except Exception:
    pass
print("Ready.")

## 2. Settings and playbook

In [ ]:
#@title Settings { run: "auto" }
SYMBOL = "NQ"                      #@param {type:"string"}
EQUITY = 50000                     #@param {type:"number"}
RISK_PCT = 1.0                     #@param {type:"number"}
VALUE_PER_POINT = 0                #@param {type:"number"}
SIZE_STEP = 1                      #@param {type:"number"}
MODEL = "claude-opus-5-5"          #@param ["claude-opus-5-5", "claude-sonnet-5-5"] {allow-input: true}
EFFORT = "high"                    #@param ["low", "medium", "high", "xhigh", "max"]
ASOF = ""                          #@param {type:"string"}
PLAYBOOK_DOC_ID = "1_Lb_UD-FKp7NTl6yqI9zD9g3csx0Imj9PPV99NlOqKQ"  #@param {type:"string"}
TZ = "America/New_York"            #@param {type:"string"}
USE_JOURNAL = False                #@param {type:"boolean"}
# SYMBOL: NQ, MNQ, ES, MES, YM, RTY, GC, CL (or any Yahoo ticker). VALUE_PER_POINT: 0 = use the standard value for the symbol
#   (NQ 20, MNQ 2, ES 50, MES 5). RISK_PCT: R93 default 1.0, 0.5 while learning, 2.0 is the hard ceiling.
# USE_JOURNAL: optional. Tick it to apply the loss-streak sizing rules R95/R96 from a trade log (needs JOURNAL to be a FILE, not a folder).
# SIZE_STEP: smallest tradable size (1 contract). ASOF: blank = analyse the latest completed candle; or e.g. 2025-03-12 08:45
#   (New York time) to replay a past moment - Yahoo only keeps about 60 days of 5m/15m data.

import io, re
from pathlib import Path
from google.colab import auth, drive as gdrive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

gdrive.mount("/content/drive")
ICT = Path("/content/drive/MyDrive/ict")
SHOTS_DIR, DATA_DIR, OUT_DIR, JOURNAL = ICT / "screenshots", ICT / "data", ICT / "ideas", ICT / "journal.json"
ECON_DIR = ICT / "economic"
for d in (SHOTS_DIR, DATA_DIR, OUT_DIR, ECON_DIR):
    d.mkdir(parents=True, exist_ok=True)

from ict_agent.market_data import point_value
if not VALUE_PER_POINT:
    VALUE_PER_POINT = point_value(SYMBOL)
    assert VALUE_PER_POINT, f"No standard point value for {SYMBOL}: set VALUE_PER_POINT yourself (value of a 1.0 price move per contract)"

auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
meta = drive.files().get(fileId=PLAYBOOK_DOC_ID, fields="name,mimeType,modifiedTime").execute()
if meta["mimeType"] == "application/vnd.google-apps.document":
    try:
        data = drive.files().export(fileId=PLAYBOOK_DOC_ID, mimeType="text/markdown").execute()
    except Exception:
        data = drive.files().export(fileId=PLAYBOOK_DOC_ID, mimeType="text/plain").execute()
else:                                                   # an uploaded .md/.txt file
    buf = io.BytesIO(); dl = MediaIoBaseDownload(buf, drive.files().get_media(fileId=PLAYBOOK_DOC_ID))
    done = False
    while not done: _, done = dl.next_chunk()
    data = buf.getvalue()
PLAYBOOK = data.decode("utf-8") if isinstance(data, bytes) else data
n_rules = len(set(re.findall(r"\bR\d{1,3}\b", PLAYBOOK)))
print(f"Playbook '{meta['name']}' (edited {meta['modifiedTime'][:16]}): {len(PLAYBOOK):,} chars, {n_rules} distinct rule numbers")
if n_rules < 100:
    print("WARNING: expected about 122 rules. The export may be missing tabs/sections - check the doc before trusting results.")
print(f"Point value {VALUE_PER_POINT:g}; risk {RISK_PCT}% of {EQUITY:,} = {EQUITY * RISK_PCT / 100:,.0f} per trade")
print(f"Put chart screenshots in {SHOTS_DIR}")
print(f"Put the weekly economic calendar CSV in {ECON_DIR} (times in New York time)")

## 3. Run
Press run for your trade idea. **DRY_RUN is a free preview only - it does NOT produce a trade idea.** Leave it **unticked** to get the idea (this calls Claude, a few cents). Tick it only to check your data first. The newest screenshots in the folder are used (up to `MAX_SHOTS`); tick `SHOW_PROMPT` to also see the raw data sent.

In [ ]:
DRY_RUN = False   #@param {type:"boolean"}
MAX_SHOTS = 6     #@param {type:"integer"}
SHOW_PROMPT = False  #@param {type:"boolean"}

import pandas as pd
from IPython.display import Markdown, display
from ict_agent.analyze import prepare, run_analysis
from ict_agent.data import load_timeframes
from ict_agent.economic import load_calendar
from ict_agent.images import load_folder
from ict_agent.market_data import fetch_yahoo
from ict_agent.risk import RiskConfig
from ict_agent.trade_card import card_markdown, make_card, report_markdown

# 1. candles: your CSVs if present, otherwise Yahoo
if list(DATA_DIR.glob("*.csv")):
    frames = load_timeframes(DATA_DIR, TZ, derive=["1h", "4h", "1d"])
    print(f"Using CSV files from {DATA_DIR}: {list(frames)}")
else:
    market = fetch_yahoo(SYMBOL, TZ)
    frames = market.frames
    print(f"Downloaded {market.ticker} candles from Yahoo: {list(frames)}; newest completed candle ended "
          f"{market.last_bar_end:%Y-%m-%d %H:%M} ({market.age_minutes:.0f} min ago)")
    for w in market.warnings:
        print(" -", w)
for tf, df in frames.items():
    print(f"  {tf:>3}: {len(df):>6} bars, last {df.index[-1]:%Y-%m-%d %H:%M}, close {df['close'].iloc[-1]:.2f}")

# 2. screenshots
IMAGES, SHOT_NAMES, skipped = load_folder(SHOTS_DIR, MAX_SHOTS)
print(f"{len(IMAGES)} screenshot(s): {SHOT_NAMES}" + (f"  (skipped unreadable: {skipped})" if skipped else ""))
if not IMAGES:
    print("No screenshots found - the analysis will use the candle data only.")

# 3. economic calendar (CSV exports in the economic folder)
calendar = load_calendar(ECON_DIR, TZ)
if calendar.available:
    print(f"Economic calendar: {len(calendar.df)} events from {', '.join(calendar.files)}")
else:
    print("No economic calendar loaded - the news rules (R64-R66) cannot be checked.")
for m in calendar.messages:
    print(" -", m)

# 4. analyse
asof = pd.Timestamp(ASOF, tz=TZ) if ASOF.strip() else None
RISK = RiskConfig(EQUITY, VALUE_PER_POINT, RISK_PCT, SIZE_STEP)   # raises if RISK_PCT is above the 2% ceiling
journal_path = None
if USE_JOURNAL and JOURNAL.is_dir():
    print(f"WARNING: {JOURNAL} is a folder, not a file - continuing without the trade journal (rename the folder or pick another file name).")
elif USE_JOURNAL:
    journal_path = JOURNAL
else:
    print("Trade journal is off: the loss-streak rules R95/R96 are not applied (every trade is sized at the full risk % above).")
if DRY_RUN:
    features, system, content = prepare(frames, SYMBOL, PLAYBOOK, "", IMAGES, asof, SHOT_NAMES, calendar)
    t = features["time"]
    eco = features["economic"]
    if not eco["available"]:
        eco_line = "none loaded - news rules R64-R66 NOT checked"
    else:
        eco_line = f"{eco['events_in_calendar']} events; high impact today: {', '.join(eco['flags']['high_impact_today']) or 'none'}"
        if eco["news_hold"]:
            eco_line += f"; **HOLD ({eco['news_hold']['rule']}): {eco['news_hold']['reason']}**"
        eco_line += "".join(f"; warning: {w}" for w in eco["warnings"])
    lines = [
        "## 👀 DRY RUN - preview only, no trade idea yet",
        "> Nothing was sent to Claude and nothing was charged. This just shows the data that **would** be sent.",
        "> **To get your trade idea: untick `DRY_RUN` (above) and run this cell again.**",
        "",
        f"- **Market:** {SYMBOL} at {features['current_price']} as of {t['asof']} ({t['day_of_week']}, {t['clock']}, {t['current_session']})",
        f"- **Trading window:** {t['trading_window']['status'].replace('_', ' ')}   |   Day note: {t['day_guidance']}",
        f"- **Candles:** {', '.join(f'{k} ({len(v)} bars)' for k, v in frames.items())}",
        f"- **Screenshots:** {len(IMAGES)} ({', '.join(SHOT_NAMES) or 'none'})   |   **Economic calendar:** {eco_line}",
        f"- **Playbook:** {len(PLAYBOOK):,} characters, {n_rules} rule numbers",
        f"- **Risk:** {RISK_PCT}% of {EQUITY:,} = {EQUITY * RISK_PCT / 100:,.0f} per trade",
    ]
    display(Markdown("\n".join(lines)))
    if SHOW_PROMPT:
        print(content[-1]["text"][:6000])
else:
    result, features = run_analysis(frames, SYMBOL, PLAYBOOK, "", IMAGES, asof, MODEL, EFFORT, OUT_DIR, RISK, journal_path, SHOT_NAMES, calendar)
    card = make_card(result, features)
    display(Markdown(card_markdown(card)))
    display(Markdown("---\n" + report_markdown(result, features, str(OUT_DIR))))

## 4. Log a closed trade (keeps R95/R96 working)
After a trade closes, record its result so the next run halves or restores risk correctly. `RISK_AMOUNT` is what you planned to risk on it; `PNL` is negative for a loss.

In [ ]:
#@title Log a closed trade { run: "auto" }
PNL = 0.0           #@param {type:"number"}
RISK_AMOUNT = 0.0   #@param {type:"number"}
LOG_IT = False      #@param {type:"boolean"}

from ict_agent.risk import log_trade, load_journal, risk_state, RiskConfig
if not USE_JOURNAL:
    print('Trade journal is off (tick USE_JOURNAL in the settings cell to use it).')
elif LOG_IT and RISK_AMOUNT > 0:
    log_trade(JOURNAL, PNL, RISK_AMOUNT, SYMBOL, RISK_PCT)
    print("Logged. Untick LOG_IT so it is not logged twice.")
if USE_JOURNAL:
  state = risk_state(load_journal(JOURNAL), RiskConfig(EQUITY, VALUE_PER_POINT, RISK_PCT, SIZE_STEP))
  print(f"{state['trades_in_journal']} trades in journal -> next trade risk {state['effective_risk_pct']}%")
  for r in state["reasons"]: print(" -", r)